# 📄 Azure AI Document Intelligence — Lab AI-102

**Objectif**: Maîtriser l'extraction d'informations depuis des documents structurés et non-structurés.

## Compétences AI-102 couvertes
- Utiliser les modèles pré-construits (read, layout, invoice, receipt, idDocument)
- Extraire du texte, des tableaux et des paires clé-valeur
- Comprendre les scores de confiance
- Traiter des documents depuis des bytes ou des URLs
- Analyser des documents multi-pages et multi-langues

In [ ]:
# Installation des dépendances
%pip install azure-ai-documentintelligence python-dotenv -q

In [ ]:
import os
from dotenv import load_dotenv
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest
from azure.core.credentials import AzureKeyCredential

load_dotenv('../.env')

endpoint = os.getenv('AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT')
key = os.getenv('AZURE_DOCUMENT_INTELLIGENCE_KEY')

client = DocumentIntelligenceClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key)
)
print('✅ Client Document Intelligence initialisé')

## 1. Modèle `prebuilt-read` — Extraction de texte générale

In [ ]:
# Analyser un document depuis une URL publique
# AI-102: prebuilt-read extrait le texte de n'importe quel document

sample_url = "https://raw.githubusercontent.com/Azure/azure-sdk-for-python/main/sdk/documentintelligence/azure-ai-documentintelligence/samples/sample_forms/forms/Invoice_1.pdf"

poller = client.begin_analyze_document(
    model_id="prebuilt-read",
    analyze_request=AnalyzeDocumentRequest(url_source=sample_url)
)
result = poller.result()

print(f"Pages analysées: {len(result.pages)}")
print(f"\nContenu extrait:\n{result.content[:500]}...")

# Langues détectées
if result.languages:
    print(f"\nLangues: {[lang.locale for lang in result.languages]}")

## 2. Modèle `prebuilt-layout` — Tableaux et structure

In [ ]:
# AI-102: prebuilt-layout détecte la structure: tableaux, paragraphes, rôles

poller = client.begin_analyze_document(
    model_id="prebuilt-layout",
    analyze_request=AnalyzeDocumentRequest(url_source=sample_url)
)
result = poller.result()

# Afficher les tableaux
if result.tables:
    print(f"Tableaux détectés: {len(result.tables)}")
    for i, table in enumerate(result.tables):
        print(f"\nTableau {i+1}: {table.row_count} lignes × {table.column_count} colonnes")
        # Afficher les premières cellules
        for cell in table.cells[:6]:
            print(f"  [{cell.row_index},{cell.column_index}] {cell.content}")
else:
    print("Aucun tableau détecté")

# Afficher les paragraphes avec leur rôle
if result.paragraphs:
    print(f"\nParagraphes: {len(result.paragraphs)}")
    for para in result.paragraphs[:3]:
        role = para.role or 'body'
        print(f"  [{role}] {para.content[:80]}")

## 3. Modèle `prebuilt-invoice` — Extraction de factures

In [ ]:
# AI-102: prebuilt-invoice extrait les champs spécifiques aux factures
# Champs: VendorName, CustomerName, InvoiceDate, DueDate, InvoiceTotal, etc.

invoice_url = "https://raw.githubusercontent.com/Azure/azure-sdk-for-python/main/sdk/documentintelligence/azure-ai-documentintelligence/samples/sample_forms/forms/Invoice_1.pdf"

poller = client.begin_analyze_document(
    model_id="prebuilt-invoice",
    analyze_request=AnalyzeDocumentRequest(url_source=invoice_url)
)
result = poller.result()

if result.documents:
    for doc in result.documents:
        print(f"Type de document: {doc.doc_type}")
        print(f"Confiance: {doc.confidence:.2%}\n")
        
        # Champs extraits
        fields_to_show = [
            'VendorName', 'CustomerName', 'InvoiceId',
            'InvoiceDate', 'DueDate', 'InvoiceTotal', 'SubTotal'
        ]
        for field_name in fields_to_show:
            if doc.fields and field_name in doc.fields:
                field = doc.fields[field_name]
                confidence = f" ({field.confidence:.0%})" if field.confidence else ""
                print(f"  {field_name}: {field.content}{confidence}")

## 4. Extraction de paires clé-valeur

In [ ]:
# AI-102: Les paires clé-valeur sont détectées automatiquement dans les formulaires

import pandas as pd

# Utiliser le résultat invoice précédent
if result.key_value_pairs:
    kv_data = []
    for kv in result.key_value_pairs:
        kv_data.append({
            'Clé': kv.key.content if kv.key else '',
            'Valeur': kv.value.content if kv.value else '',
            'Confiance': f"{kv.confidence:.0%}" if hasattr(kv, 'confidence') and kv.confidence else 'N/A'
        })
    
    df = pd.DataFrame(kv_data)
    print("Paires Clé-Valeur extraites:")
    print(df.to_string(index=False))
else:
    print("Aucune paire clé-valeur détectée")

## 5. Analyse d'un fichier local

In [ ]:
# AI-102: Analyser un fichier local (bytes)
# Décommenter et adapter le chemin vers votre fichier

# with open('mon_document.pdf', 'rb') as f:
#     content = f.read()

# poller = client.begin_analyze_document(
#     model_id="prebuilt-read",
#     analyze_request=content,
#     content_type="application/pdf"
# )
# result = poller.result()
# print(result.content)

print("Décommentez le code ci-dessus avec votre fichier local")

## Résumé AI-102 — Document Intelligence

| Modèle | Usage principal | Champs clés |
|--------|----------------|-------------|
| `prebuilt-read` | Extraction de texte | content, pages, languages |
| `prebuilt-layout` | Structure + tableaux | tables, paragraphs, pages |
| `prebuilt-invoice` | Factures | VendorName, InvoiceTotal, DueDate |
| `prebuilt-receipt` | Reçus | MerchantName, Total, Items |
| `prebuilt-idDocument` | Pièces d'identité | FirstName, LastName, DateOfBirth |
| `prebuilt-businessCard` | Cartes de visite | ContactNames, Emails, PhoneNumbers |